In [1]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\fahee\AppData\Local\Temp\ipykernel_38696\4288846553.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\FAHEEM\My_Programs\RAG_Beginners\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

if os.environ['GOOGLE_API_KEY']:
    print("Google api key is set")
else:
    raise ValueError("Gemini API key is not set")


llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", google_api_key=api_key, temperature=0.7)

Google api key is set


In [17]:
loader = PyMuPDFLoader("./docs/pdf/29_Paper.pdf")
pdf_loader = loader.load()

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(pdf_loader)

In [19]:
embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2553.93it/s]


In [20]:
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding=embedding 
)

retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)

In [21]:
from typing import TypedDict, List
from langchain_core.documents import Document

class graph_schema(TypedDict):
    question:str
    documents: List[Document]
    answer: str

In [24]:
from langchain_core.prompts import ChatPromptTemplate

def retirver(state: graph_schema) -> graph_schema:
    question = state['question']
    document = retriever.invoke(question)
    return {
        "documents": document
    }

def generator(state: graph_schema) -> graph_schema:
    question = state['question']
    document = state['documents']

    context = '\n\n'.join(doc.page_content for doc in document)

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are a helpful assistant. Answer the question using ONLY the provided context. If the answer is not in the context just say 'I don't know' Context:{context}"),
        ("human", f"Here's a question to ask for you {question}")
    ])

    chain = prompt | llm

    result = chain.invoke({"context": context, "question": question})
    return {
        "result": result
    }
